# Teste isolado — AGERGS (Notícias)

Fonte candidata: **AGERGS - Agência Estadual de Regulação dos Serviços
Públicos Delegados do Rio Grande do Sul**, setor Transporte. Já existia
placeholder em `controle_fontes` (`source_id='—'`, `status='Não
iniciada'`, `importancia_original='Média'`). Notebook **descartável**
(Fase 1) — sem dispatcher, sem `atualizar_status_fonte`, sem gravar nada
em produção.

URL fornecida: `https://agergs.rs.gov.br/inicial`. Página de notícias
identificada por navegação: `/noticias`.

## ⚠️ Aviso operacional importante — não usar GET no endpoint AJAX

Durante a investigação (fora deste notebook, via `curl` isolado), uma
tentativa de **GET** com querystring no endpoint
`/_service/conteudo/pagedlistfilho` (em vez de POST) causou um **reset
de conexão TLS**, e em seguida o domínio inteiro (`agergs.rs.gov.br`)
passou a dar timeout de conexão por vários minutos, mesmo para a home
pública -- sugere bloqueio temporário de IP/rede reagindo a tráfego
anômalo nesse endpoint interno. As células abaixo usam **somente POST**
com o header `X-Requested-With: XMLHttpRequest`, que respondeu
normalmente em todas as tentativas -- não trocar por GET, e não repetir
chamadas em sequência rápida.

In [ ]:
%pip install --quiet httpx beautifulsoup4 lxml
dbutils.library.restartPython()

In [ ]:
import urllib.parse
import httpx
from bs4 import BeautifulSoup

BASE = "https://agergs.rs.gov.br"
URL_NOTICIAS = f"{BASE}/noticias"
URL_AJAX = f"{BASE}/_service/conteudo/pagedlistfilho"
ID_NOTICIAS = "11547"

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)
HEADERS_PAGINA = {"User-Agent": USER_AGENT, "Accept-Language": "pt-BR,pt;q=0.9"}
HEADERS_AJAX = {**HEADERS_PAGINA, "X-Requested-With": "XMLHttpRequest"}

## Teste 1 — robots.txt e a página pública /noticias

`robots.txt` permite tudo (`Allow: /`). A página pública carrega
normalmente (200), mas o corpo da listagem
(`.matriz-ui-pagedlist-body`) vem **vazio** no HTML estático -- os itens
são carregados via AJAX (framework "Matriz", usado por outros órgãos do
governo do RS).

In [ ]:
with httpx.Client(headers=HEADERS_PAGINA, timeout=30, follow_redirects=True) as client:
    resp_robots = client.get(f"{BASE}/robots.txt")
    resp_noticias = client.get(URL_NOTICIAS)

print(f"/robots.txt -> HTTP {resp_robots.status_code}\n{resp_robots.text}\n")
print(f"/noticias   -> HTTP {resp_noticias.status_code}, {len(resp_noticias.text)} chars")

soup_noticias = BeautifulSoup(resp_noticias.text, "lxml")
corpo_lista = soup_noticias.select_one(".matriz-ui-pagedlist-body")
print(f"corpo da listagem estática: {'vazio' if not corpo_lista or not corpo_lista.get_text(strip=True) else 'tem conteúdo'}")

## Teste 2 — endpoint AJAX (POST, nunca GET)

`GET` nesse endpoint devolve HTTP 400 ("Serviço Indisponível") -- só
aceita `POST` com header `X-Requested-With: XMLHttpRequest`. Os nomes
dos campos do formulário (`currentPage`, `pageSize`) vieram das próprias
mensagens de erro do endpoint ao tentar nomes errados primeiro
(`pagenumber`/`pagesize` -> "Campo X não encontrado na coleção.").
`pageSize=50` cobre o histórico inteiro numa chamada só (9 itens em
18/08/2026).

In [ ]:
with httpx.Client(timeout=30) as client:
    resp_ajax = client.post(
        URL_AJAX,
        params={"id": ID_NOTICIAS, "templatename": "pagina.listapagina.padrao"},
        data={"ordem": "RECENTES", "currentPage": "1", "pageSize": "50"},
        headers=HEADERS_AJAX,
    )

print(f"HTTP {resp_ajax.status_code}, content-type={resp_ajax.headers.get('content-type')}")
dados = resp_ajax.json()
print(f"recordcount={dados['recordcount']}, pagecount={dados['pagecount']}")

## Teste 3 — parsear o fragmento HTML do campo `body` e listar os itens

In [ ]:
soup_itens = BeautifulSoup(dados["body"], "lxml")
itens = []
for artigo in soup_itens.find_all("article"):
    tag_titulo = artigo.find("h2", class_="conteudo-lista__item__titulo")
    tag_a = tag_titulo.find("a", href=True) if tag_titulo else None
    if not tag_a:
        continue
    titulo = tag_a.get_text(" ", strip=True)
    url_absoluta = urllib.parse.urljoin(URL_NOTICIAS, tag_a["href"].strip())
    tag_time = artigo.find("time")
    published_at = tag_time["datetime"][:10] if tag_time and tag_time.get("datetime") else None
    itens.append({"titulo": titulo, "url": url_absoluta, "published_at": published_at})

print(f"{len(itens)} itens extraídos (esperado: {dados['recordcount']}).\n")
for item in itens:
    print(f"{item['published_at'] or '?':<12} {item['titulo'][:80]}")

## Teste 4 — abrir uma notícia e extrair o texto completo

Página do artigo usa CMS "Matriz": título em `h1.artigo__titulo`,
subtítulo em `p.artigo__subtitulo`, data em
`p.artigo__pubdate time[datetime]`, texto completo em `.artigo__texto`
(já adicionado a `SELETORES_CONTEUDO` do dispatcher genérico).

**Cuidado com `extrair_titulo_h1` genérico**: a página tem **3** `<h1>`
-- o primeiro é um masthead institucional oculto (`class="text-hide"`,
"Agência Estadual de Regulação..."), sem relação com o artigo. Usar
`soup.find("h1")` ingenuamente pegaria esse masthead errado -- mesmo bug
já documentado em AESBE/ABRACE. Por isso o dispatcher usa
`extrair_titulo=None` para AGERGS (mantém o título já correto vindo da
listagem).

In [ ]:
with httpx.Client(headers=HEADERS_PAGINA, timeout=30, follow_redirects=True) as client:
    resp_artigo = client.get(itens[0]["url"])

soup_artigo = BeautifulSoup(resp_artigo.text, "lxml")
todos_h1 = [h1.get_text(" ", strip=True) for h1 in soup_artigo.find_all("h1")]
print(f"total de <h1> na página: {len(todos_h1)}")
for i, texto_h1 in enumerate(todos_h1):
    print(f"  [{i}] {texto_h1[:90]}")

texto_artigo = soup_artigo.select_one(".artigo__texto")
print(f"\n.artigo__texto encontrado: {texto_artigo is not None}")
if texto_artigo:
    texto_limpo = texto_artigo.get_text("\n", strip=True)
    print(f"tamanho do texto: {len(texto_limpo)} chars\n")
    print(texto_limpo[:600])

## Conclusão da Fase 1

Os quatro testes passam: `robots.txt` permite tudo, o endpoint AJAX
(POST + `X-Requested-With`) devolve a listagem completa (9 itens,
título/data/link prontos), e a página de cada notícia tem texto completo
limpo em `.artigo__texto` -- sem paywall, sem WAF, sem Selenium.

**Avaliação para a Fase 2**: encaixa no dispatcher genérico
`ingest-scraping` -- é "baixar uma URL e extrair itens de forma padrão",
mesmo que a listagem em si precise de uma chamada POST própria em vez do
`baixar_pagina()` genérico (GET). `listar_agergs()` já foi adicionado ao
dispatcher: ignora o `html` genérico recebido (que vem só de um GET em
`/noticias`, usado apenas para o loop principal não falhar) e faz a
própria chamada POST ao endpoint AJAX. `.artigo__texto` foi adicionado a
`SELETORES_CONTEUDO`, e `extrair_titulo=None` evita o bug do masthead
duplo. Histórico pequeno (9 itens) -- sem paginação necessária,
`pageSize=50` cobre tudo numa chamada só.